# 투구 제구 성공 확률 모델 V2.2 학습

V1 Legacy Global과 고정된 R/F Expert만 결합합니다. 2024 R 성능 보호 조건이 있는 가중치 탐색과 보수적 walk-forward 보정을 적용하며 Pitcher Expert는 실험 코드에만 남깁니다.

In [1]:
import importlib.util
import importlib.metadata
import subprocess
import sys

required_packages = {
    "catboost": ("catboost", "1.2.10"),
    "sklearn": ("scikit-learn", "1.8.0"),
    "pyarrow": ("pyarrow", "25.0.1"),
}
install = []
for module, (distribution, wanted) in required_packages.items():
    found = importlib.util.find_spec(module) is not None
    current = importlib.metadata.version(distribution) if found else None
    if current != wanted:
        install.append(f"{distribution}=={wanted}")
if install:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *install])


In [2]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display
from train_v22 import run_training

ROOT = Path.cwd().resolve()
if not (ROOT / "result.py").exists():
    raise FileNotFoundError("프로젝트 루트에서 main.ipynb를 실행해 주세요.")
RUN_DIR = Path(os.environ.get("BASEBALL_RUN_DIR", ROOT)).resolve()
print(f"project={ROOT}, run_dir={RUN_DIR}")
print(f"task_type={os.environ.get('BASEBALL_TASK_TYPE', 'GPU')}, devices={os.environ.get('BASEBALL_GPU_DEVICES', '0')}")


project=D:\baseball, run_dir=D:\baseball
task_type=GPU, devices=0


## V2.2 학습 및 OOF 평가

기본값은 전체 데이터와 GPU 0번입니다. 빠른 점검만 할 때는 실행 전에 `BASEBALL_FAST_MODE=1`을 설정하세요. FAST_MODE 결과는 제출에 사용하면 안 됩니다.

In [3]:
result = run_training(ROOT, RUN_DIR)
display(result["metrics"].sort_values(["season", "model"]))
print("Selected R expert:", result["summary"]["selected_regular"])
print("Selected F expert:", result["summary"]["selected_futures"])
print("Final weights by type:", result["summary"]["ensemble"]["weight_map_by_game_type"])
print("Calibration:", result["summary"]["calibration"])
display(result["blend_grid"])
display(result["iteration_sensitivity"])
summary_keys = ["weighted_cv_brier", "brier_2024", "brier_2024_r", "brier_2024_f", "mean_gap_2024_r", "mean_gap_2024_f", "worst_fold_brier", "calibration_slope", "calibration_intercept"]
display(pd.Series({key: result["summary"][key] for key in summary_keys}, name="V2.2"))
print("Acceptance:", result["summary"]["acceptance"], "all_passed=", result["summary"]["all_acceptance_passed"])


Default metric period is 5 because BrierScore is/are not implemented for GPU
Metric BrierScore is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.2495552	test: 0.2494742	best: 0.2494742 (0)	total: 121ms	remaining: 2m 25s
100:	learn: 0.2428164	test: 0.2434661	best: 0.2434658 (98)	total: 10.9s	remaining: 1m 58s
200:	learn: 0.2417944	test: 0.2434978	best: 0.2434590 (109)	total: 21.6s	remaining: 1m 47s
bestTest = 0.2434590418
bestIteration = 109
Shrink model to first 110 iterations.
0:	learn: 0.6927225	test: 0.6929383	best: 0.6929383 (0)	total: 91ms	remaining: 1m 49s
100:	learn: 0.6842276	test: 0.6903740	best: 0.6903740 (100)	total: 9.8s	remaining: 1m 46s
200:	learn: 0.6820413	test: 0.6904889	best: 0.6903694 (105)	total: 19.4s	remaining: 1m 36s
bestTest = 0.6903693543
bestIteration = 105
Shrink model to first 106 iterations.
0:	learn: 0.6873660	test: 0.6898985	best: 0.6898985 (0)	total: 38.2ms	remaining: 45.8s
100:	learn: 0.6063815	test: 0.6141262	best: 0.6141262 (100)	total: 3.77s	remaining: 41s
200:	learn: 0.5941103	test: 0.6132891	best: 0.6131136 (193)	total: 7.65s	remaining: 38s
300:	learn: 0.5834898	test: 0.6130031	

Default metric period is 5 because BrierScore is/are not implemented for GPU
Metric BrierScore is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	learn: 0.2495285	test: 0.2499824	best: 0.2499824 (0)	total: 142ms	remaining: 2m 49s
100:	learn: 0.2425751	test: 0.2530932	best: 0.2499770 (1)	total: 14.1s	remaining: 2m 33s
bestTest = 0.2499769812
bestIteration = 1
Shrink model to first 2 iterations.
0:	learn: 0.6928024	test: 0.6929855	best: 0.6929855 (0)	total: 117ms	remaining: 2m 20s
100:	learn: 0.6852640	test: 0.6912697	best: 0.6911746 (72)	total: 13s	remaining: 2m 21s
bestTest = 0.6911746352
bestIteration = 72
Shrink model to first 73 iterations.
0:	learn: 0.6863259	test: 0.6931475	best: 0.6931475 (0)	total: 48.1ms	remaining: 57.7s
100:	learn: 0.5965903	test: 0.7390444	best: 0.6931475 (0)	total: 4.36s	remaining: 47.5s
bestTest = 0.6931474557
bestIteration = 0
Shrink model to first 1 iterations.
0:	learn: 0.6922263	total: 82.7ms	remaining: 3.22s
39:	learn: 0.6807641	total: 1.61s	remaining: 0us
0:	learn: 0.6922263	total: 74.9ms	remaining: 4.42s
59:	learn: 0.6798478	total: 2.26s	remaining: 0us
0:	learn: 0.6922263	total: 70.8ms	rema

Default metric period is 5 because BrierScore is/are not implemented for GPU
Metric BrierScore is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


...

Selected R expert: regular_hl1_5
Selected F expert: futures_post2023
Final weights by type: {'R': {'legacy_global': 1.0, 'game_type': 0.0}, 'F': {'legacy_global': 0.75, 'game_type': 0.25}}
Calibration: {'version': 4, 'method': 'affine', 'selection_statistics': {'identity': {'weighted_cv_brier': 0.24772110879421233, 'brier_2024': 0.24807550013065338, 'brier_2024_r': 0.2481723427772522, 'brier_2024_f': 0.24735410511493683}, 'affine': {'weighted_cv_brier': 0.2476910501718521, 'brier_2024': 0.2479894906282425, 'brier_2024_r': 0.24810990691184998, 'brier_2024_f': 0.24709278345108032}}, 'walk_forward_folds': [{'season': 2022, 'slope': 1.0, 'intercept': 0.0}, {'season': 2023, 'slope': 1.1814131802135557, 'intercept': -0.1}, {'season': 2024, 'slope': 1.1568325189832573, 'intercept': -0.0842063447616812}], 'trained_on_seasons': [2022, 2023, 2024], 'slope': 1.1469227207481136, 'intercept': -0.08019920634797717}


,game_type,expert_weight,weighted_brier,aggressive_weighted_brier,legacy_brier_2024,v1_reference_brier_2024_r,brier_2022,brier_2023,brier_2024,protects_legacy_anchor,protects_2024_r
0,R,0.00,0.248762,0.248717,0.248172,0.248099,0.248626,0.249837,0.248172,True,False
1,R,0.10,0.248710,0.248668,0.248172,0.248099,0.248610,0.249654,0.248184,False,False
2,R,0.20,0.248669,0.248629,0.248172,0.248099,0.248598,0.249493,0.248203,False,False
3,R,0.25,0.248652,0.248614,0.248172,0.248099,0.248593,0.249421,0.248214,False,False
4,R,0.30,0.248638,0.248602,0.248172,0.248099,0.248589,0.249355,0.248228,False,False
5,R,0.35,0.248627,0.248592,0.248172,0.248099,0.248586,0.249294,0.248243,False,False
6,R,0.40,0.248618,0.248586,0.248172,0.248099,0.248583,0.249239,0.248259,False,False
7,F,0.20,0.240331,0.244377,0.247446,0.248099,0.206905,0.250893,0.247364,True,True
8,F,0.25,0.240328,0.244362,0.247446,0.248099,0.207014,0.250826,0.247354,True,True
9,F,0.30,0.240330,0.244351,0.247446,0.248099,0.207139,0.250761,0.247348,True,True


,family,season,iterations,brier
0,legacy_global,2023,40,0.252669
1,legacy_global,2023,60,0.253116
2,legacy_global,2023,80,0.253361
3,legacy_global,2023,100,0.253442
4,legacy_global,2023,120,0.253477
5,legacy_global,2024,40,0.248231
6,legacy_global,2024,60,0.248096
7,legacy_global,2024,80,0.248056
8,legacy_global,2024,100,0.248026
9,legacy_global,2024,120,0.248007


weighted_cv_brier        0.247691
brier_2024               0.247989
brier_2024_r             0.248110
brier_2024_f             0.247093
mean_gap_2024_r          0.002797
mean_gap_2024_f          0.005241
worst_fold_brier         0.249984
calibration_slope        1.146923
calibration_intercept   -0.080199
Name: V2.2, dtype: float64

Acceptance: {'weighted_cv_brier': False, 'brier_2024': True, 'brier_2024_f': True, 'brier_2024_r': False} all_passed= False


In [4]:
display(result["correlations"])
display(result["importance"].head(25))
display(pd.read_csv(result["submission_path"]))
print("다음으로 evaluation.ipynb를 실행해 세부 평가를 확인하세요.")


,legacy_global,game_type
legacy_global,1.000000,0.857367
game_type,0.857367,1.000000


,feature,legacy_global,regular_expert,futures_expert,mean_importance
98,season,27.065938,3.910802,0.000000,10.325580
35,game_type,23.708667,0.000000,0.000000,7.902889
95,same_hand,3.945284,5.356313,7.220696,5.507431
14,asof_pitcher_prev5_game_success_rate,4.036794,5.397322,4.580135,4.671417
17,asof_pitcher_success_rate,8.176593,3.376800,2.034391,4.529261
62,pitcher_id,0.000000,8.660007,2.863092,3.841033
36,hand_matchup_code,2.254183,3.194209,5.891728,3.780040
3,asof_pitcher_ball_rate,3.264302,5.255553,2.056177,3.525344
15,asof_pitcher_reverse_rate,2.928139,4.643882,2.641890,3.404637
27,batter_team_id,4.314533,3.542629,2.010326,3.289163


,row_id,control_success
0,TEST_000001,0.448190
1,TEST_000017,0.436853
2,TEST_000213,0.455871
3,TEST_005332,0.491633
4,TEST_035185,0.470772


다음으로 evaluation.ipynb를 실행해 세부 평가를 확인하세요.
